# 02 — Download dos Dados ABIDE

Este notebook realiza o download e a organização dos dados do
ABIDE utilizados no estudo de classificação de Transtorno do
Espectro Autista (TEA).

O processamento e a análise dos dados serão realizados em notebooks
posteriores.

**Objetivo:** Baixar o ABIDE I (Preprocessed Connectomes Project, CC200) e organizar fenótipos e séries temporais.

**Entradas:** Internet (somente se os dados ainda não existirem em `Data/`).

**Processamento:** `fetch_abide_pcp` (cpac, band-pass, sem GSR, `rois_cc200`, quality_checked). O download é pulado se os artefatos finais já existirem.

**Saídas:** `Data/raw/ABIDE_pcp/`, `Data/raw/phenotypic_abide.csv`, `Data/roi_timeseries/roi_timeseries_abide.pkl`.

In [ ]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

In [3]:
import pickle
import pandas as pd
import nilearn
import nibabel as nib

print("Nilearn:", nilearn.__version__)
print("NiBabel:", nib.__version__)

from nilearn.datasets import fetch_abide_pcp

Download ABIDE I

In [9]:
from nilearn.datasets import fetch_abide_pcp

# Verificação condicional: se os artefatos finais já existem, não baixa de novo.
_csv_fenotipico = PATHS['raw'] / 'phenotypic_abide.csv'
_pkl_series = PATHS['roi_timeseries'] / 'roi_timeseries_abide.pkl'

if _csv_fenotipico.exists() and _pkl_series.exists():
    abide = None
    print('Dados já existem — download ignorado:')
    print(' ', _csv_fenotipico)
    print(' ', _pkl_series)
else:
    abide = fetch_abide_pcp(
        data_dir=str(PATHS['raw']),
        n_subjects=None,
        pipeline="cpac",
        band_pass_filtering=True,
        global_signal_regression=False,
        derivatives=["rois_cc200"],
        quality_checked=True
    )

    print("Download concluído!")
    print("Total de sujeitos:", len(abide.phenotypic))

[fetch_abide_pcp] Dataset directory found: <BASE_DIR>/Data/raw/ABIDE_pcp

Download concluído!
Total de sujeitos: 871


Levou 22 minutos para baixar 871 sujeitos


In [13]:
# Metadados fenotípicos: diagnóstico (DX_GROUP), site (SITE_ID), idade, sexo etc.
# DX_GROUP: 1 = TEA, 2 = Controle
if abide is None:
    phenotypic = pd.read_csv(PATHS['raw'] / 'phenotypic_abide.csv')
    print('phenotypic_abide.csv já existe — usando o arquivo existente.')
else:
    phenotypic = pd.DataFrame(abide.phenotypic)
    phenotypic.to_csv(os.path.join(PATHS['raw'], 'phenotypic_abide.csv'), index=False)

print(phenotypic[['SUB_ID', 'SITE_ID', 'DX_GROUP', 'AGE_AT_SCAN', 'SEX']].head())
print('\nDistribuição por grupo diagnóstico (1=TEA, 2=Controle):')
print(phenotypic['DX_GROUP'].value_counts())

   SUB_ID SITE_ID  DX_GROUP  AGE_AT_SCAN  SEX
1   50003    PITT         1        24.45    1
2   50004    PITT         1        19.09    1
3   50005    PITT         1        13.73    2
4   50006    PITT         1        13.37    1
5   50007    PITT         1        17.78    1

Distribuição por grupo diagnóstico (1=TEA, 2=Controle):
DX_GROUP
2    468
1    403
Name: count, dtype: int64


Temos 871 sujeitos para cada sujeito temos 196 momentos no tempo
        ×
200 regiões cerebrais (ROIs)

In [14]:
# --------------------------------------------------------
# Salva as séries temporais ROI (uma matriz [tempo x 200 ROIs] por sujeito)
# em Data/roi_timeseries/, indexadas por SUB_ID — este é o formato de entrada
# do notebook 04 (cálculo de conectividade).
# --------------------------------------------------------
out_path = os.path.join(PATHS['roi_timeseries'], 'roi_timeseries_abide.pkl')

if abide is None:
    print('roi_timeseries_abide.pkl já existe — nada a salvar:', out_path)
else:
    roi_timeseries = {}
    for sub_id, ts in zip(phenotypic['SUB_ID'], abide.rois_cc200):
        roi_timeseries[sub_id] = ts

    with open(out_path, 'wb') as f:
        pickle.dump(roi_timeseries, f)

    print(f'{len(roi_timeseries)} séries temporais salvas em: {out_path}')
    print('Formato de uma série temporal (exemplo):', next(iter(roi_timeseries.values())).shape)

871 séries temporais salvas em: <BASE_DIR>/Data/roi_timeseries/roi_timeseries_abide.pkl
Formato de uma série temporal (exemplo): (196, 200)
